<div style="background: linear-gradient(136deg, #1a3a5c 0%, #2d6a9f 100%); padding: 40px 32px 32px 32px; border-radius: 12px; margin-bottom: 8px;">
  <h1 style="color: #ffffff; font-size: 2.0em; margin: 0 0 6px 0; font-family: 'Segoe UI', sans-serif; font-weight: 700;">
    🗂️ Week 4, Lesson 7 — Advanced Pandas, Plotting &amp; LAS Files
  </h1>
  <h2 style="color: #a8d4f5; font-size: 1.2em; margin: 0 0 18px 0; font-family: 'Segoe UI', sans-serif; font-weight: 400;">
    Sorting &middot; Merging &middot; Pivot Tables &middot; apply() 
&middot; Histograms &middot; Box Plots &middot; Multi-track Logs &middot; Lasio
  </h2>
  <hr style="border: 1px solid rgba(255,255,255,0.25); margin: 16px 0;">
  <table style="color: #cce4ff; font-family: 'Segoe UI', sans-serif; font-size: 0.95em;">
    <tr>
      <td style="padding: 3px 24px 3px 0;"><strong>📅 Week:</strong></td><td>4 (Lesson 7) — Advanced Pandas, Plotting &amp; LAS Files</td>
      <td style="padding: 3px 24px 3px 32px;"><strong>⏱️ Duration:</strong></td><td>40 Minutes</td>
    </tr>
    <tr>
      <td style="padding: 3px 24px 3px 0;"><strong>🎯 Phase:</strong></td><td>Month 1 — Foundations</td>
      <td style="padding: 3px 24px 3px 32px;"><strong>👤 Audience:</strong></td><td>Engineers · Geoscientists · Researchers</td>
    </tr>
    <tr>
      <td style="padding: 3px 24px 3px 0;"><strong>🧑‍🏫 Instructor:</strong></td><td>Dr. Daniel Wamriew</td>
      <td style="padding: 3px 24px 3px 32px;"><strong>✉️ Contact:</strong></td><td>wamriewdan@gmail.com</td>
    </tr>
  </table>
</div>

<div style="border-left: 4px solid #2ca87f; background: #f0faf5; 
padding: 16px 20px; border-radius: 6px; margin: 12px 0;">
<h3 style="color: #1a7a5c; margin: 0 0 10px 0;">How to Use This Notebook</h3>
<ul style="margin: 0; padding-left: 20px; color: #2d4a3e; font-size: 0.95em;">
<li>Run cells in order using <strong>Shift + Enter</strong></li>
<li>Read each explanation cell before running the code below it</li>
<li>Complete each <strong>Student Activity</strong> before moving on</li>
<li>You will need <strong>lasio</strong> for Section 9 -- 
install it once with: <code>pip install lasio</code></li>
<li>The file <code>well_log_data.csv</code> must be in the same folder as this notebook</li>
</ul>
</div>

## Contents

1. [Why This Lesson](#sec1-why)
2. [Setup -- Load & Clean Data](#sec2-setup)
3. [Sorting & Ranking](#sec3-sort)
4. [Merging DataFrames](#sec4-merge)
5. [Pivot Tables](#sec5-pivot)
6. [Applying Functions Row-by-Row](#sec6-apply)
7. [Histograms -- Log Distribution Analysis](#sec7-hist)
8. [Box Plots -- Comparing Formations & Wells](#sec8-box)
9. [Multi-track Well Log Display](#sec9-multitrack)
10. [Introduction to LAS Files with lasio](#sec10-las)
11. [Practice Exercise](#sec11-practice)
12. [Recap & Homework](#sec12-recap)

<a id='sec1-why'></a>
## 1. Why This Lesson

<div style="border-left: 4px solid #2d6a9f; background: #eef5fc; 
padding: 16px 20px; border-radius: 6px; margin: 8px 0;">
<h4 style="color: #1a3a5c; margin: 0 0 10px 0;">Building on Lesson 6</h4>
<p style="color: #2d3a4a; margin: 0 0 10px 0;">In Lesson 6 you learned how to load CSV data, 
handle missing values, filter rows, compute new columns, group data, and make basic plots. 
That foundation covers most day-to-day data exploration tasks.</p>
<p style="color: #2d3a4a; margin: 0 0 10px 0;">In this lesson you go further:</p>
<ul style="color: #2d3a4a; margin: 0 0 10px 0; padding-left: 20px;">
<li><strong>Sorting &amp; merging</strong> -- rank wells, combine tables</li>
<li><strong>Pivot tables</strong> -- cross-tabulate logs by well and formation</li>
<li><strong>apply()</strong> -- classify every row using your own Python function</li>
<li><strong>Histograms &amp; box plots</strong> -- understand log distributions</li>
<li><strong>Multi-track log display</strong> -- the standard petrophysical visualization</li>
<li><strong>LAS files with lasio</strong> -- the universal well-log file format</li>
</ul>
<p style="color: #2d3a4a; margin: 0;">By the end of this lesson you can wrangle, classify, 
visualize, and export real petrophysical datasets in the formats used by industry.</p>
</div>

<a id='sec2-setup'></a>
## 2. Setup -- Load &amp; Clean Data

We start where Lesson 6 finished: load the CSV and apply forward-fill cleaning 
so every section of this notebook has a clean DataFrame to work with.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Load the well log dataset
df = pd.read_csv("../data/well_log_data.csv")

# Apply forward-fill within each well (best practice for depth series)
log_cols = ["GR_API", "NPHI_frac", "RHOB_gcc", "RT_ohmm", "SW_frac"]
df[log_cols] = df.groupby("Well_ID")[log_cols].ffill()
df[log_cols] = df.groupby("Well_ID")[log_cols].bfill()

# Add derived columns we will use throughout the lesson
df["Porosity_pct"] = df["NPHI_frac"] * 100
df["Density_diff"] = df["RHOB_gcc"] - 2.65   # deviation from quartz grain density

print("Dataset shape:", df.shape)
print("Columns:", list(df.columns))
df.head()

<a id='sec3-sort'></a>
## 3. Sorting &amp; Ranking DataFrames

<div style="border-left: 4px solid #2d6a9f; background: #eef5fc; 
padding: 16px 20px; border-radius: 6px; margin: 8px 0;">
<h4 style="color: #1a3a5c; margin: 0 0 8px 0;">Key methods</h4>
<ul style="color: #2d3a4a; margin: 0; padding-left: 20px;">
<li><code>df.sort_values("col")</code> -- sort ascending by one column</li>
<li><code>df.sort_values(["col1", "col2"], ascending=[True, False])</code> -- multi-level sort</li>
<li><code>df.nlargest(n, "col")</code> -- top-N rows by a column (descending)</li>
<li><code>df.nsmallest(n, "col")</code> -- bottom-N rows by a column</li>
<li><code>df["col"].rank()</code> -- assign a rank number to every row</li>
</ul>
</div>

In [ ]:
# Sort all rows by GR (low GR = cleaner reservoir)
df_sorted = df.sort_values("GR_API")
print("Cleanest (low GR) rows:")
print(df_sorted[["Well_ID", "Depth_m", "Formation", "GR_API"]].head(15))

In [ ]:
# Multi-level sort: by Well, then by Depth
df_by_well_depth = df.sort_values(["Well_ID", "Depth_m"], ascending=[True, True])
print(df_by_well_depth[["Well_ID", "Depth_m", "Formation"]].head(10))

In [ ]:
# Find the 10 rows with the highest resistivity (best hydrocarbon indicators)
top_rt = df.nlargest(10, "RT_ohmm")
print("Top 10 highest resistivity readings:")
print(top_rt[["Well_ID", "Depth_m", "Formation", "RT_ohmm", "SW_frac"]])

In [ ]:
# Rank each row by porosity (per well group)
df["Porosity_rank"] = df.groupby("Well_ID")["NPHI_frac"].rank(ascending=False)

# Show the best-porosity row per well
best_por = df[df["Porosity_rank"] == 1][["Well_ID", "Depth_m", "Formation", "NPHI_frac"]]
print("Best porosity depth per well:")
print(best_por.to_string(index=False))

<div style="border-left: 4px solid #f39c12; background: #fdf6e3; 
padding: 16px 20px; border-radius: 6px; margin: 12px 0;">
<h4 style="color: #b7770d; margin: 0 0 10px 0;">Student Activity 1 -- Sorting</h4>
<p style="color: #5d4037; margin: 0 0 8px 0;"><strong>Track A (Engineering):</strong> 
Sort <code>df</code> by <code>SW_frac</code> ascending, then by <code>RT_ohmm</code> descending. 
Print the top 5 rows and explain what these rows represent geologically.</p>
<p style="color: #5d4037; margin: 0;"><strong>Track B (All Students):</strong> 
Use <code>nlargest()</code> to find the 5 rows with the highest GR values. 
Which well and formation are they from?</p>
</div>

In [ ]:
# Your code here


<a id='sec4-merge'></a>
## 4. Merging DataFrames

<div style="border-left: 4px solid #2d6a9f; background: #eef5fc; 
padding: 16px 20px; border-radius: 6px; margin: 8px 0;">
<h4 style="color: #1a3a5c; margin: 0 0 8px 0;">Why merge?</h4>
<p style="color: #2d3a4a; margin: 0 0 10px 0;">In real projects, well data lives in multiple tables. 
Log data (depth-by-depth measurements) is stored separately from well header data 
(location, spud date, operator). <code>pd.merge()</code> joins them on a shared key column.</p>
<table style="border-collapse: collapse; width: 100%; font-size: 0.9em; color: #2d3a4a;">
<tr style="background: #d0e8f7;">
<th style="padding: 6px 10px; border: 1px solid #a8c8e8; text-align: left;">how=</th>
<th style="padding: 6px 10px; border: 1px solid #a8c8e8;">Rows kept</th>
</tr>
<tr><td style="padding: 6px 10px; border: 1px solid #a8c8e8;"><code>'inner'</code></td>
<td style="padding: 6px 10px; border: 1px solid #a8c8e8;">Only rows matching in both tables</td></tr>
<tr style="background: #f5f9fd;"><td style="padding: 6px 10px; border: 1px solid #a8c8e8;">
<code>'left'</code></td>
<td style="padding: 6px 10px; border: 1px solid #a8c8e8;">All rows from left table; NaN where no match</td></tr>
<tr><td style="padding: 6px 10px; border: 1px solid #a8c8e8;"><code>'outer'</code></td>
<td style="padding: 6px 10px; border: 1px solid #a8c8e8;">All rows from both tables</td></tr>
</table>
</div>

In [ ]:
# Create a well header table (metadata about each well)
well_header = pd.DataFrame({
    "Well_ID": ["Well_A", "Well_B", "Well_C", "Well_D"],
    "Operator": ["AfroPetro", "NileOil", "SaharaE&P", "AfroPetro"],
    "Surface_X": [412500, 413100, 411800, 412900],
    "Surface_Y": [218400, 218900, 217700, 219200],
    "Spud_Year": [2019, 2020, 2021, 2022],
    "KB_Elevation_m": [210.5, 214.2, 208.8, 212.1]
})
print("Well header table:")
print(well_header)

In [ ]:
# Inner join: attach header info to every log row
df_merged = pd.merge(df, well_header, on="Well_ID", how="inner")
print("Merged shape:", df_merged.shape)
print("New columns:", [c for c in df_merged.columns if c not in list(df.columns)])
df_merged[["Well_ID", "Depth_m", "Formation", "Operator", "Spud_Year"]].head(8)

In [ ]:
# Group by operator -- how many rows per operator?
op_counts = df_merged.groupby("Operator")["Depth_m"].count().reset_index()
op_counts.columns = ["Operator", "Log_Rows"]

# Average GR per operator
op_gr = df_merged.groupby("Operator")["GR_API"].mean().reset_index()
op_gr.columns = ["Operator", "Mean_GR"]

# Merge these two summaries together
op_summary = pd.merge(op_counts, op_gr, on="Operator")
op_summary["Mean_GR"] = op_summary["Mean_GR"].round(1)
print(op_summary)

<div style="border-left: 4px solid #f39c12; background: #fdf6e3; 
padding: 16px 20px; border-radius: 6px; margin: 12px 0;">
<h4 style="color: #b7770d; margin: 0 0 10px 0;">Student Activity 2 -- Merging</h4>
<p style="color: #5d4037; margin: 0 0 8px 0;"><strong>Track A:</strong> 
Create a second small DataFrame called <code>formation_info</code> with columns 
<code>Formation</code>, <code>Type</code> (e.g., 'Seal', 'Reservoir', 'Basement'), and 
<code>Avg_Thickness_m</code>. Merge it with <code>df</code> on <code>Formation</code>. 
How many rows does the merged DataFrame have?</p>
<p style="color: #5d4037; margin: 0;"><strong>Track B:</strong> 
Using <code>df_merged</code>, calculate the mean porosity for each <code>Operator</code>. 
Which operator has the highest average porosity?</p>
</div>

In [ ]:
# Your code here


<a id='sec5-pivot'></a>
## 5. Pivot Tables

<div style="border-left: 4px solid #2d6a9f; background: #eef5fc; 
padding: 16px 20px; border-radius: 6px; margin: 8px 0;">
<h4 style="color: #1a3a5c; margin: 0 0 8px 0;">What is a pivot table?</h4>
<p style="color: #2d3a4a; margin: 0 0 8px 0;">A pivot table is a cross-tabulation: rows represent one 
category (e.g., Well), columns represent another (e.g., Formation), and cells hold a 
summary statistic (e.g., mean GR). This is one of the fastest ways to compare properties 
across two categorical dimensions simultaneously.</p>
<code style="background: #ddeeff; padding: 4px 8px; border-radius: 4px; display: block; 
margin-top: 8px; color: #1a3a5c;">
pd.pivot_table(df, values="GR_API", index="Well_ID", columns="Formation", aggfunc="mean")
</code>
</div>

In [ ]:
# Mean GR by Well x Formation
pt_gr = pd.pivot_table(
    df,
    values="GR_API",
    index="Well_ID",
    columns="Formation",
    aggfunc="mean"
).round(1)
print("Mean GR (API) -- Well x Formation:")
print(pt_gr)

In [ ]:
# Mean resistivity by Well x Formation
pt_rt = pd.pivot_table(
    df,
    values="RT_ohmm",
    index="Well_ID",
    columns="Formation",
    aggfunc="mean"
).round(1)
print("Mean Resistivity (ohm.m) -- Well x Formation:")
print(pt_rt)

In [ ]:
# Count of rows per well x formation (useful for coverage check)
pt_count = pd.pivot_table(
    df,
    values="Depth_m",
    index="Well_ID",
    columns="Formation",
    aggfunc="count"
)
print("Row count per Well x Formation:")
print(pt_count)

<div style="border-left: 4px solid #f39c12; background: #fdf6e3; 
padding: 16px 20px; border-radius: 6px; margin: 12px 0;">
<h4 style="color: #b7770d; margin: 0 0 10px 0;">Student Activity 3 -- Pivot Tables</h4>
<p style="color: #5d4037; margin: 0 0 8px 0;"><strong>Track A:</strong> 
Build a pivot table of <strong>mean SW_frac</strong> by Well x Formation. 
Which well-formation combination shows the lowest water saturation (best hydrocarbon potential)?</p>
<p style="color: #5d4037; margin: 0;"><strong>Track B:</strong> 
Build a pivot table of <strong>mean NPHI_frac</strong> by Well x Formation. 
Which formation has the highest average porosity across all wells?</p>
</div>

In [ ]:
# Your code here


<a id='sec6-apply'></a>
## 6. Applying Functions Row-by-Row

<div style="border-left: 4px solid #2d6a9f; background: #eef5fc; 
padding: 16px 20px; border-radius: 6px; margin: 8px 0;">
<h4 style="color: #1a3a5c; margin: 0 0 8px 0;">df.apply() -- your function on every row</h4>
<p style="color: #2d3a4a; margin: 0 0 8px 0;">When a computed column requires logic that spans 
multiple input columns, <code>apply()</code> lets you write a plain Python function and apply it 
to each row. Use <code>axis=1</code> to pass each row as a Series to your function.</p>
<ul style="color: #2d3a4a; margin: 0; padding-left: 20px;">
<li><code>df.apply(my_func, axis=1)</code> -- call <code>my_func(row)</code> for every row</li>
<li>Access columns inside the function as <code>row["GR_API"]</code></li>
<li>Return a single value -- it becomes the new column value for that row</li>
</ul>
</div>

In [ ]:
# Reservoir Quality (RQ) classification using multiple log inputs

def classify_rq(row):
    """Classify reservoir quality from GR, NPHI, RT, and SW."""
    gr = row["GR_API"]
    nphi = row["NPHI_frac"]
    rt = row["RT_ohmm"]
    sw = row["SW_frac"]

    if gr < 50 and nphi > 0.20 and rt > 50 and sw < 0.40:
        return "Excellent"
    elif gr < 75 and nphi > 0.15 and rt > 20 and sw < 0.60:
        return "Good"
    elif gr < 100 and nphi > 0.10:
        return "Fair"
    else:
        return "Poor"

df["RQ_Class"] = df.apply(classify_rq, axis=1)

print("Reservoir Quality class counts:")
print(df["RQ_Class"].value_counts())

In [ ]:
# How does RQ distribute per well?
rq_well = pd.pivot_table(
    df,
    values="Depth_m",
    index="Well_ID",
    columns="RQ_Class",
    aggfunc="count",
    fill_value=0
)
print("RQ class counts per well:")
print(rq_well)

In [ ]:
# Lithology flag using a lambda on a single column
df["Lith_Flag"] = df["GR_API"].apply(lambda g: "Sand" if g < 65 else ("Shaly_Sand" if g < 90 else "Shale"))
print("Lithology distribution:")
print(df["Lith_Flag"].value_counts())

<div style="border-left: 4px solid #f39c12; background: #fdf6e3; 
padding: 16px 20px; border-radius: 6px; margin: 12px 0;">
<h4 style="color: #b7770d; margin: 0 0 10px 0;">Student Activity 4 -- apply()</h4>
<p style="color: #5d4037; margin: 0 0 8px 0;"><strong>Track A:</strong> 
Write a function <code>fluid_type(row)</code> that returns <code>'Hydrocarbon'</code> 
if <code>SW_frac &lt; 0.5</code> and <code>RT_ohmm &gt; 30</code>, otherwise <code>'Brine'</code>. 
Apply it to create a <code>Fluid_Type</code> column. How many hydrocarbon-flagged rows are there?</p>
<p style="color: #5d4037; margin: 0;"><strong>Track B:</strong> 
Use <code>apply()</code> with a lambda to create a column <code>RHOB_class</code> that is 
<code>'Dense'</code> if <code>RHOB_gcc &gt; 2.5</code>, else <code>'Porous'</code>.</p>
</div>

In [ ]:
# Your code here


<a id='sec7-hist'></a>
## 7. Histograms -- Log Distribution Analysis

<div style="border-left: 4px solid #2d6a9f; background: #eef5fc; 
padding: 16px 20px; border-radius: 6px; margin: 8px 0;">
<h4 style="color: #1a3a5c; margin: 0 0 8px 0;">Why histograms for well logs?</h4>
<p style="color: #2d3a4a; margin: 0 0 8px 0;">A histogram shows the <em>frequency distribution</em> 
of a log. GR histograms classically show two peaks -- a low-GR sand peak and a high-GR shale peak. 
The valley between them is used to set the sand/shale cutoff.</p>
<ul style="color: #2d3a4a; margin: 0; padding-left: 20px;">
<li><code>df["GR_API"].hist(bins=30)</code> -- quick one-liner</li>
<li><code>ax.hist(data, bins=30, edgecolor='black', alpha=0.7)</code> -- more control</li>
<li>Overlay histograms for different formations to compare their distributions</li>
</ul>
</div>

In [ ]:
# GR histogram -- all data
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(df["GR_API"].dropna(), bins=40, color="#2d6a9f", edgecolor="white", alpha=0.8)
ax.axvline(65, color="#e07b39", linestyle="--", linewidth=1.5, label="Sand cutoff (65 API)")
ax.axvline(90, color="#c0392b", linestyle="--", linewidth=1.5, label="Shale cutoff (90 API)")
ax.set_xlabel("GR (API)")
ax.set_ylabel("Count")
ax.set_title("GR Distribution -- All Wells")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Overlay GR histograms by formation
formations = list(df["Formation"].unique())
colors = ["#2d6a9f", "#2ca87f", "#e07b39", "#8e44ad", "#c0392b"]

fig, ax = plt.subplots(figsize=(9, 4))
for form, col in zip(formations, colors):
    subset = df[df["Formation"] == form]["GR_API"].dropna()
    ax.hist(subset, bins=25, alpha=0.5, label=form, color=col, edgecolor="none")

ax.set_xlabel("GR (API)")
ax.set_ylabel("Count")
ax.set_title("GR Distribution by Formation")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Panel of histograms for all log columns
log_labels = ["GR (API)", "NPHI (frac)", "RHOB (g/cc)", "RT (ohm.m)", "SW (frac)"]
log_cols_plot = ["GR_API", "NPHI_frac", "RHOB_gcc", "RT_ohmm", "SW_frac"]
panel_colors = ["#2d6a9f", "#2ca87f", "#8e44ad", "#e07b39", "#c0392b"]

fig, axes = plt.subplots(1, 5, figsize=(14, 3.5))
for ax, col, label, col_color in zip(axes, log_cols_plot, log_labels, panel_colors):
    ax.hist(df[col].dropna(), bins=30, color=col_color, edgecolor="white", alpha=0.85)
    ax.set_title(label, fontsize=9)
    ax.set_ylabel("Count", fontsize=8)
    ax.tick_params(labelsize=7)

fig.suptitle("Log Distributions -- All Wells", fontsize=11, y=1.02)
plt.tight_layout()
plt.show()

<div style="border-left: 4px solid #f39c12; background: #fdf6e3; 
padding: 16px 20px; border-radius: 6px; margin: 12px 0;">
<h4 style="color: #b7770d; margin: 0 0 10px 0;">Student Activity 5 -- Histograms</h4>
<p style="color: #5d4037; margin: 0 0 8px 0;"><strong>Track A:</strong> 
Plot overlapping GR histograms for each individual well (not formation). 
Use 4 different colors. Do the wells have similar GR distributions?</p>
<p style="color: #5d4037; margin: 0;"><strong>Track B:</strong> 
Plot a histogram of <code>RHOB_gcc</code>. Add a vertical line at 2.65 g/cc (quartz density). 
What does it mean when density is below this line?</p>
</div>

In [ ]:
# Your code here


<a id='sec8-box'></a>
## 8. Box Plots -- Comparing Formations &amp; Wells

<div style="border-left: 4px solid #2d6a9f; background: #eef5fc; 
padding: 16px 20px; border-radius: 6px; margin: 8px 0;">
<h4 style="color: #1a3a5c; margin: 0 0 8px 0;">Reading a box plot</h4>
<ul style="color: #2d3a4a; margin: 0; padding-left: 20px;">
<li><strong>Box</strong> -- Q1 (25th pct) to Q3 (75th pct): the interquartile range (IQR)</li>
<li><strong>Line in box</strong> -- median (50th pct)</li>
<li><strong>Whiskers</strong> -- extend to 1.5 x IQR from the box edges</li>
<li><strong>Dots beyond whiskers</strong> -- outliers</li>
</ul>
<p style="color: #2d3a4a; margin: 8px 0 0 0;">Box plots are ideal for comparing a log across 
multiple categories (formations or wells) in a single glance.</p>
</div>

In [ ]:
# Box plot of GR by Formation
formations_ordered = ["Shale_Cap", "Reservoir_1", "Transition", "Reservoir_2", "Basement"]
gr_groups = [df[df["Formation"] == f]["GR_API"].dropna().values for f in formations_ordered]

fig, ax = plt.subplots(figsize=(9, 4))
bp = ax.boxplot(gr_groups, labels=formations_ordered, patch_artist=True,
                medianprops={"color": "black", "linewidth": 2})

box_colors = ["#8e44ad", "#2ca87f", "#e07b39", "#2d6a9f", "#c0392b"]
for patch, color in zip(bp["boxes"], box_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.axhline(65, color="#e07b39", linestyle="--", linewidth=1, alpha=0.6, label="Sand cutoff")
ax.set_ylabel("GR (API)")
ax.set_title("GR Distribution by Formation")
ax.legend(fontsize=8)
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

In [ ]:
# Box plot of porosity by Well
wells = list(df["Well_ID"].unique())
nphi_groups = [df[df["Well_ID"] == w]["NPHI_frac"].dropna().values for w in wells]

fig, ax = plt.subplots(figsize=(7, 4))
bp2 = ax.boxplot(nphi_groups, labels=wells, patch_artist=True,
                 medianprops={"color": "black", "linewidth": 2})

well_colors = ["#2d6a9f", "#2ca87f", "#e07b39", "#8e44ad"]
for patch, color in zip(bp2["boxes"], well_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_ylabel("NPHI (frac)")
ax.set_title("Porosity Distribution by Well")
plt.tight_layout()
plt.show()

In [ ]:
# Water saturation by formation -- spot the reservoir!
sw_groups = [df[df["Formation"] == f]["SW_frac"].dropna().values for f in formations_ordered]

fig, ax = plt.subplots(figsize=(9, 4))
bp3 = ax.boxplot(sw_groups, labels=formations_ordered, patch_artist=True,
                 medianprops={"color": "black", "linewidth": 2})
for patch in bp3["boxes"]:
    patch.set_facecolor("#a8d4f5")
    patch.set_alpha(0.8)

ax.axhline(0.5, color="#c0392b", linestyle="--", linewidth=1.2, label="SW=0.5 threshold")
ax.set_ylabel("SW (frac)")
ax.set_title("Water Saturation by Formation")
ax.legend(fontsize=8)
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

<div style="border-left: 4px solid #f39c12; background: #fdf6e3; 
padding: 16px 20px; border-radius: 6px; margin: 12px 0;">
<h4 style="color: #b7770d; margin: 0 0 10px 0;">Student Activity 6 -- Box Plots</h4>
<p style="color: #5d4037; margin: 0 0 8px 0;"><strong>Track A:</strong> 
Create a 1x2 panel of box plots: left panel shows <code>RT_ohmm</code> by formation, 
right panel shows <code>RHOB_gcc</code> by formation. 
Describe in 2 sentences what each plot tells you about the reservoir intervals.</p>
<p style="color: #5d4037; margin: 0;"><strong>Track B:</strong> 
Create a box plot of <code>GR_API</code> grouped by well. 
Do all wells sample the same GR range?</p>
</div>

In [ ]:
# Your code here


<a id='sec9-multitrack'></a>
## 9. Multi-track Well Log Display

<div style="border-left: 4px solid #2d6a9f; background: #eef5fc; 
padding: 16px 20px; border-radius: 6px; margin: 8px 0;">
<h4 style="color: #1a3a5c; margin: 0 0 8px 0;">The standard petrophysical plot</h4>
<p style="color: #2d3a4a; margin: 0 0 8px 0;">In petrophysics, logs are displayed as <strong>depth vs. 
log value</strong> -- depth on the Y-axis increasing downward. Multiple tracks (columns) show 
different logs side by side so you can correlate lithology changes across tracks.</p>
<p style="color: #2d3a4a; margin: 0;">A standard display includes: 
Track 1 -- GR &nbsp;|&nbsp; Track 2 -- RHOB &amp; NPHI overlaid 
&nbsp;|&nbsp; Track 3 -- RT (log scale) &nbsp;|&nbsp; Track 4 -- SW</p>
</div>

In [ ]:
# Select one well and sort by depth
well = "Well_A"
wd = df[df["Well_ID"] == well].sort_values("Depth_m")

depth = wd["Depth_m"].values

fig, axes = plt.subplots(1, 4, figsize=(12, 8), sharey=True)
fig.suptitle(f"Multi-track Log Display -- {well}", fontsize=13, fontweight="bold")

# Track 1: GR
axes[0].plot(wd["GR_API"], depth, color="#2ca87f", linewidth=1)
axes[0].axvline(65, color="#e07b39", linestyle="--", linewidth=0.8, alpha=0.6)
axes[0].set_xlabel("GR (API)")
axes[0].set_title("GR", fontsize=9)
axes[0].set_xlim(0, 200)
axes[0].invert_yaxis()
axes[0].set_ylabel("Depth (m)")

# Track 2: RHOB and NPHI overlaid (density-porosity crossover)
ax2a = axes[1]
ax2b = ax2a.twiny()
ax2a.plot(wd["RHOB_gcc"], depth, color="#c0392b", linewidth=1, label="RHOB")
ax2b.plot(wd["NPHI_frac"], depth, color="#2d6a9f", linewidth=1, linestyle="--", label="NPHI")
ax2a.set_xlabel("RHOB (g/cc)", color="#c0392b")
ax2b.set_xlabel("NPHI (frac)", color="#2d6a9f")
axes[1].set_title("RHOB / NPHI", fontsize=9)
ax2a.set_xlim(1.95, 2.95)
ax2b.set_xlim(0.45, -0.15)   # reversed: NPHI and RHOB should crossover in gas

# Track 3: RT (log scale)
axes[2].semilogx(wd["RT_ohmm"], depth, color="#8e44ad", linewidth=1)
axes[2].axvline(20, color="#e07b39", linestyle="--", linewidth=0.8, alpha=0.6)
axes[2].set_xlabel("RT (ohm.m)")
axes[2].set_title("Resistivity", fontsize=9)

# Track 4: SW
axes[3].plot(wd["SW_frac"], depth, color="#1a3a5c", linewidth=1)
axes[3].axvline(0.5, color="#c0392b", linestyle="--", linewidth=0.8, alpha=0.6)
axes[3].set_xlabel("SW (frac)")
axes[3].set_title("Water Saturation", fontsize=9)
axes[3].set_xlim(0, 1)

for ax in axes:
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

<div style="border-left: 4px solid #f39c12; background: #fdf6e3; 
padding: 16px 20px; border-radius: 6px; margin: 12px 0;">
<h4 style="color: #b7770d; margin: 0 0 10px 0;">Student Activity 7 -- Multi-track Plot</h4>
<p style="color: #5d4037; margin: 0 0 8px 0;"><strong>Track A:</strong> 
Reproduce the 4-track display for <strong>Well_B</strong>. 
Identify the depth intervals where you would expect to find hydrocarbons based on the logs. 
Add horizontal lines at those depths.</p>
<p style="color: #5d4037; margin: 0;"><strong>Track B:</strong> 
Plot a 2-track display (GR and SW) for any one well. 
Invert the Y-axis so depth increases downward. What pattern do you see?</p>
</div>

In [ ]:
# Your code here


<a id='sec10-las'></a>
## 10. Introduction to LAS Files with lasio

<div style="border-left: 4px solid #2d6a9f; background: #eef5fc; 
padding: 16px 20px; border-radius: 6px; margin: 8px 0;">
<h4 style="color: #1a3a5c; margin: 0 0 8px 0;">What is a LAS file?</h4>
<p style="color: #2d3a4a; margin: 0 0 8px 0;">LAS (Log ASCII Standard) is the universal file format 
for well logs. Every petrophysical software package reads and writes LAS. A LAS file has two parts:</p>
<ul style="color: #2d3a4a; margin: 0 0 8px 0; padding-left: 20px;">
<li><strong>Header sections</strong> -- metadata: well name, UWI, location, log curve descriptions, units</li>
<li><strong>Data section</strong> -- depth + log values as space-separated columns, one row per depth step</li>
</ul>
<p style="color: #2d3a4a; margin: 0;">The <strong>lasio</strong> library reads LAS files into Python 
and converts them to Pandas DataFrames with one method call: <code>las.df()</code>.</p>
</div>

### What a LAS file looks like

```
~VERSION ---------------------------------------------------
 VERS.                 2.0 : CWLS LOG ASCII STANDARD - VERSION 2.0
 WRAP.                 NO  : ONE LINE PER DEPTH STEP
~WELL ------------------------------------------------------
 WELL.                 WELL_A    : Well name
 UWI .                 10-010-01234-00-00 : Unique Well Identifier
 STRT.M                2000.0  : Start depth
 STOP.M                2090.0  : Stop depth
 STEP.M                0.5     : Depth step
 NULL.                 -999.25 : Null value
~CURVE ---------------------------------------------------------
 DEPT.M              : Depth
 GR  .API            : Gamma Ray
 NPHI.V/V            : Neutron Porosity
 RHOB.G/C3           : Bulk Density
 RT  .OHMM           : True Resistivity
 SW  .V/V            : Water Saturation
~A ----------------------------------------------------------
2000.0  112.3  0.115  2.541  3.21  0.981
2000.5   98.7  0.121  2.518  4.10  0.972
 ...
```

<div style="border-left: 4px solid #2ca87f; background: #f0faf5; 
padding: 14px 20px; border-radius: 6px; margin: 8px 0;">
<h4 style="color: #1a7a5c; margin: 0 0 6px 0;">Installing lasio</h4>
<p style="color: #2d4a3e; margin: 0;">If you have not installed lasio yet, run this once 
in your terminal (not a notebook cell):</p>
<code style="display: block; background: #1a3a5c; color: #a8d4f5; padding: 8px 12px; 
border-radius: 4px; margin-top: 8px;">pip install lasio</code>
</div>

In [ ]:
# Step 1: Create a synthetic LAS file for Well_A
# We write a LAS 2.0 file from scratch using Well_A data from our DataFrame

well_a = df[df["Well_ID"] == "Well_A"].sort_values("Depth_m").reset_index(drop=True)

las_lines = []
las_lines.append("~VERSION ---------------------------------------------------")
las_lines.append(" VERS.                 2.0 : CWLS LOG ASCII STANDARD - VERSION 2.0")
las_lines.append(" WRAP.                 NO  : ONE LINE PER DEPTH STEP")
las_lines.append("~WELL ------------------------------------------------------")
las_lines.append(" WELL.                 WELL_A    : Well name")
las_lines.append(" UWI .                 10-010-10001-00-00 : Unique Well Identifier")
las_lines.append(f" STRT.M                {well_a['Depth_m'].min():.1f}  : Start depth")
las_lines.append(f" STOP.M                {well_a['Depth_m'].max():.1f}  : Stop depth")
las_lines.append(" STEP.M                0.5     : Depth step")
las_lines.append(" NULL.                 -999.25 : Null value")
las_lines.append(" COMP.                 AfroPetro : Operator")
las_lines.append(" FLD .                 Nile Basin : Field")
las_lines.append("~CURVE ---------------------------------------------------------")
las_lines.append(" DEPT.M              : Measured Depth")
las_lines.append(" GR  .API            : Gamma Ray")
las_lines.append(" NPHI.V/V            : Neutron Porosity (fraction)")
las_lines.append(" RHOB.G/C3           : Bulk Density")
las_lines.append(" RT  .OHMM           : True Resistivity")
las_lines.append(" SW  .V/V            : Water Saturation (fraction)")
las_lines.append("~A ----------------------------------------------------------")

for _, row in well_a.iterrows():
    gr   = row["GR_API"]   if not pd.isna(row["GR_API"])   else -999.25
    nphi = row["NPHI_frac"] if not pd.isna(row["NPHI_frac"]) else -999.25
    rhob = row["RHOB_gcc"]  if not pd.isna(row["RHOB_gcc"])  else -999.25
    rt   = row["RT_ohmm"]  if not pd.isna(row["RT_ohmm"])  else -999.25
    sw   = row["SW_frac"]  if not pd.isna(row["SW_frac"])  else -999.25
    las_lines.append(f"{row['Depth_m']:8.1f}  {gr:8.3f}  {nphi:8.4f}  {rhob:8.4f}  {rt:10.3f}  {sw:8.4f}")

las_text = "\n".join(las_lines)

with open("Well_A.las", "w") as f:
    f.write(las_text)

print("LAS file written: Well_A.las")
print(f"Total lines: {len(las_lines)}")
print("\nFirst 12 lines:")
print("\n".join(las_lines[:12]))

In [ ]:
import lasio

# Step 2: Read the LAS file with lasio
las = lasio.read("Well_A.las")

print("=== LAS Header Information ===")
print(f"Well name : {las.well['WELL'].value}")
print(f"Start     : {las.well['STRT'].value} {las.well['STRT'].unit}")
print(f"Stop      : {las.well['STOP'].value} {las.well['STOP'].unit}")
print(f"Step      : {las.well['STEP'].value} {las.well['STEP'].unit}")
print(f"Null value: {las.well['NULL'].value}")

print("\n=== Curve Information ===")
curve_names = list(las.keys())
for name in curve_names:
    curve = las.curves[name]
    print(f"  {name:6s}  unit={curve.unit:6s}  desc={curve.descr}")

In [ ]:
# Step 3: Convert LAS to a Pandas DataFrame
df_las = las.df()
df_las.index.name = "Depth_m"    # the index is the depth column
df_las = df_las.reset_index()

# Replace null values (-999.25) with NaN
df_las = df_las.replace(-999.25, float("nan"))

print("LAS DataFrame shape:", df_las.shape)
print("Columns:", list(df_las.columns))
df_las.head(8)

In [ ]:
# Step 4: Plot directly from the LAS DataFrame -- same workflow as CSV!
df_las_clean = df_las.sort_values("Depth_m")

fig, axes = plt.subplots(1, 3, figsize=(9, 7), sharey=True)
fig.suptitle("Well_A -- Loaded from LAS", fontsize=12, fontweight="bold")

axes[0].plot(df_las_clean["GR"], df_las_clean["Depth_m"], color="#2ca87f", linewidth=1)
axes[0].set_xlabel("GR (API)")
axes[0].set_title("GR")
axes[0].invert_yaxis()
axes[0].set_ylabel("Depth (m)")

axes[1].plot(df_las_clean["RHOB"], df_las_clean["Depth_m"], color="#c0392b", linewidth=1)
axes[1].set_xlabel("RHOB (g/cc)")
axes[1].set_title("Density")

axes[2].plot(df_las_clean["RT"], df_las_clean["Depth_m"], color="#8e44ad", linewidth=1)
axes[2].set_xlabel("RT (ohm.m)")
axes[2].set_title("Resistivity")

for ax in axes:
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

<div style="border-left: 4px solid #2ca87f; background: #f0faf5; 
padding: 14px 20px; border-radius: 6px; margin: 8px 0;">
<h4 style="color: #1a7a5c; margin: 0 0 8px 0;">Where to get real LAS files</h4>
<ul style="color: #2d4a3e; margin: 0; padding-left: 20px;">
<li><strong>Kansas Geological Survey:</strong> kgs.ku.edu/Magellan/Well_Logs/ -- free public LAS downloads</li>
<li><strong>Equinor Volve dataset:</strong> open industry dataset with full LAS files for 24 wells</li>
<li><strong>USGS Energy Resources:</strong> energy.usgs.gov -- US basin data with LAS exports</li>
<li><strong>Petrel / Kingdom:</strong> export any well in LAS 2.0 format from the software</li>
</ul>
</div>

<div style="border-left: 4px solid #f39c12; background: #fdf6e3; 
padding: 16px 20px; border-radius: 6px; margin: 12px 0;">
<h4 style="color: #b7770d; margin: 0 0 10px 0;">Student Activity 8 -- LAS Files</h4>
<p style="color: #5d4037; margin: 0 0 8px 0;"><strong>Track A:</strong> 
Adapt the LAS file creation code to write a LAS file for <strong>Well_B</strong>. 
Read it back with lasio and confirm the shape of the resulting DataFrame. 
Then calculate mean GR and mean SW from the LAS DataFrame.</p>
<p style="color: #5d4037; margin: 0;"><strong>Track B:</strong> 
Using <code>df_las</code> (the Well_A LAS DataFrame), filter rows where <code>GR &lt; 65</code> 
(sand). How many sand rows are there? What is the average resistivity in sand intervals?</p>
</div>

In [ ]:
# Your code here


<a id='sec11-practice'></a>
## 11. Practice Exercise

<div style="border-left: 4px solid #2d6a9f; background: #eef5fc; 
padding: 16px 20px; border-radius: 6px; margin: 8px 0;">
<h4 style="color: #1a3a5c; margin: 0 0 10px 0;">End-to-End Mini-Project: Reservoir Screening Report</h4>
<p style="color: #2d3a4a; margin: 0 0 10px 0;">Using everything from this lesson, complete the steps below 
in order. They mirror a real petrophysical screening workflow.</p>
<ol style="color: #2d3a4a; margin: 0; padding-left: 22px;">
<li>Load <code>well_log_data.csv</code>, clean missing values with ffill</li>
<li>Classify every row using <code>classify_rq()</code> from Section 6</li>
<li>Build a pivot table of <code>RQ_Class == 'Excellent'</code> row counts by Well x Formation</li>
<li>For each well, find the depth range (min, max) of Excellent rows using <code>groupby()</code> and <code>sort_values()</code></li>
<li>Plot a 3-panel histogram of GR, NPHI, and RT for <em>only</em> the Excellent rows</li>
<li>Write a LAS file for the well with the most Excellent rows; read it back and confirm</li>
</ol>
</div>

In [ ]:
# Step 1: Load and clean
df_pr = pd.read_csv("well_log_data.csv")
log_cols = ["GR_API", "NPHI_frac", "RHOB_gcc", "RT_ohmm", "SW_frac"]
df_pr[log_cols] = df_pr.groupby("Well_ID")[log_cols].ffill()
df_pr[log_cols] = df_pr.groupby("Well_ID")[log_cols].bfill()

# Step 2: Classify RQ
df_pr["RQ_Class"] = df_pr.apply(classify_rq, axis=1)

# Step 3: Pivot table of Excellent rows
df_exc = df_pr[df_pr["RQ_Class"] == "Excellent"]
pt_exc = pd.pivot_table(
    df_exc, values="Depth_m", index="Well_ID",
    columns="Formation", aggfunc="count", fill_value=0
)
print("Excellent RQ rows per Well x Formation:")
print(pt_exc)

# Step 4: Depth range of Excellent rows per well
depth_range = df_exc.groupby("Well_ID")["Depth_m"].agg(["min", "max"])
depth_range.columns = ["Top_m", "Base_m"]
print("\nExcellent reservoir depth range per well:")
print(depth_range)

# Step 5: Histogram panel for Excellent rows only
fig, axes = plt.subplots(1, 3, figsize=(10, 3.5))
for ax, col, label, clr in zip(
        axes,
        ["GR_API", "NPHI_frac", "RT_ohmm"],
        ["GR (API)", "NPHI (frac)", "RT (ohm.m)"],
        ["#2ca87f", "#2d6a9f", "#8e44ad"]):
    ax.hist(df_exc[col].dropna(), bins=20, color=clr, edgecolor="white", alpha=0.85)
    ax.set_title(label, fontsize=9)
    ax.set_ylabel("Count", fontsize=8)
fig.suptitle("Log Distributions -- Excellent RQ Intervals Only", fontsize=10, y=1.02)
plt.tight_layout()
plt.show()

# Step 6: Best well by Excellent row count
exc_counts = df_exc.groupby("Well_ID")["Depth_m"].count()
best_well = exc_counts.idxmax()
print(f"\nWell with most Excellent rows: {best_well} ({exc_counts[best_well]} rows)")
print("Writing LAS file for this well...")

<a id='sec12-recap'></a>
## 12. Recap &amp; Homework

### What we covered in Lesson 7

| Topic | Key methods | Petroleum use case |
|-------|-------------|--------------------|
| Sorting & ranking | `sort_values()`, `nlargest()`, `rank()` | Rank wells by reservoir quality |
| Merging | `pd.merge()` with `how` options | Join log data with well headers |
| Pivot tables | `pd.pivot_table()` | GR / SW / RT cross-tabulated by well and formation |
| Row-wise functions | `df.apply(func, axis=1)` | Classify RQ or lithology using multiple logs |
| Histograms | `ax.hist()`, overlay loops | Log distribution analysis, cut-off determination |
| Box plots | `ax.boxplot()` | Compare log ranges across formations and wells |
| Multi-track display | `plt.subplots()` with `sharey=True`, `invert_yaxis()` | Standard petrophysical log plot |
| LAS files | `lasio.read()`, `las.df()`, manual LAS writing | Universal well-log file format |


### Homework

<div style="border-left: 4px solid #e07b39; background: #fdf3ec; 
padding: 16px 20px; border-radius: 6px; margin: 10px 0;">
<h4 style="color: #b75a1a; margin: 0 0 10px 0;">Task 1 -- Advanced Wrangling (All Students)</h4>
<p style="color: #5d3010; margin: 0 0 8px 0;">Using <code>well_log_data.csv</code>:</p>
<ol style="color: #5d3010; margin: 0; padding-left: 22px;">
<li>Load and clean the data with ffill</li>
<li>Add a column <code>Net_Pay</code>: True if <code>GR_API &lt; 65</code>, <code>SW_frac &lt; 0.5</code>, 
and <code>NPHI_frac &gt; 0.18</code></li>
<li>Use a pivot table to count Net Pay rows by Well x Formation</li>
<li>Sort wells by total Net Pay thickness (assume 0.5 m per Net Pay row)</li>
<li>Print a summary: well name and net pay thickness in metres</li>
</ol>
</div>

<div style="border-left: 4px solid #8e44ad; background: #f8f0fc; 
padding: 16px 20px; border-radius: 6px; margin: 10px 0;">
<h4 style="color: #6a1a9a; margin: 0 0 10px 0;">Task 2 -- Visualization (All Students)</h4>
<ol style="color: #4a1070; margin: 0; padding-left: 22px;">
<li>Create a box plot of <code>RT_ohmm</code> for each well (4 boxes on one plot)</li>
<li>Create a 4-track multi-track log display for the well with the most Net Pay rows</li>
<li>Overlay a colored background on the Net Pay intervals (hint: use <code>ax.axhspan()</code>)</li>
</ol>
</div>

<div style="border-left: 4px solid #c0392b; background: #fdf0ef; 
padding: 16px 20px; border-radius: 6px; margin: 10px 0;">
<h4 style="color: #922b21; margin: 0 0 10px 0;">Task 3 -- LAS (Engineering Track)</h4>
<ol style="color: #6e1f1a; margin: 0; padding-left: 22px;">
<li>Write LAS files for all 4 wells (Well_A through Well_D)</li>
<li>Read each one back with lasio and store the DataFrames in a Python list</li>
<li>Combine all 4 LAS DataFrames into a single DataFrame using <code>pd.concat()</code> 
and add a <code>Well_ID</code> column to each before combining</li>
<li>Confirm the combined DataFrame has 720 rows (same as the original CSV)</li>
</ol>
</div>

In [ ]:
# Homework Task 1 -- Net Pay analysis
# Your code here


In [ ]:
# Homework Task 2 -- Visualization
# Your code here


In [ ]:
# Homework Task 3 (Engineering) -- LAS for all wells
# Your code here


<div style="background: linear-gradient(135deg, #1a3a5c 0%, #2d6a9f 100%); 
padding: 24px 28px; border-radius: 10px; margin-top: 16px; text-align: center;">
<h3 style="color: #ffffff; margin: 0 0 8px 0;">Lesson 7 Complete!</h3>
<p style="color: #a8d4f5; margin: 0 0 10px 0;">You can now sort, merge, pivot, apply custom functions, 
produce professional multi-track plots, and read/write LAS files -- 
the core data wrangling toolkit for a petrophysicist.</p>
<p style="color: #d0eaff; font-size: 0.9em; margin: 0;">
Next: <strong>Lesson 8 -- NumPy for Petrophysical Calculations</strong></p>
</div>